# Cold-start transfer learning: how much patient history is needed?

Test whether population transfer offers a larger advantage when little target-patient history is available. This is a **new, isolated follow-up experiment**, not a replacement for the completed full-history benchmark and not a guarantee of larger gains.

**Default:** GRU, 30-minute horizon, 12 patients, seeds 41/42/43, and **3 / 5 / 7 / 10 days / full history**. The official held-out test set stays fixed. Includes MAE, RMSE, persistence, paired patient/seed uncertainty, training histories and predictions.

**Use a GPU runtime** (your A100 is suitable). This notebook contains the runner and snapshots of the repository's XML loader/preprocessor, so no repository clone or pushed changes are required. It reads your existing OhioT1DM XML files from Drive. Previously trained full-history checkpoints cannot be reused: they may have seen target data outside the limited-history budget.

Completed pretraining and each paired patient/seed/budget job are mirrored to Drive and reused after a disconnect when the data, code, configuration and package-version fingerprint matches.


## 1. Drive and experiment configuration

The budget counts **elapsed days, including validation**, ending at the last available training CGM reading. Using the most recent history controls recency across budgets. This simulates a newly available, short patient record; it is not a prospective newly diagnosed-patient cohort.

The primary follow-up is **MAE with 3 days versus this experiment's full-history control** at 30 minutes. The other budgets and RMSE are reported, even if they show no gain. Keep these settings fixed after inspecting test results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_DATA = Path('/content/drive/MyDrive/ohiot1dm')  # contains 2018/ and 2020/
DRIVE_RESULTS = Path('/content/drive/MyDrive/bg-results')
LOCAL_ROOT = Path('/content/cold_start_followup')
SMOKE_TEST = False  # True: two patients, one seed, tiny model, one epoch; NOT paper results
MODEL = 'gru'
HORIZON_MINUTES = 30
BUDGETS_DAYS = [3, 5, 7, 10, 'full']
SEEDS = [41, 42, 43]


## 2. What is controlled

- **Four inputs:** glucose, basal insulin, bolus, carbohydrates; 60-minute input window. Predict all horizon steps, score only the final step.
- **Source pretraining:** all other patients' official training records; target patient excluded. Each source is split chronologically 80/20. Source-only validation selects the checkpoint. The same checkpoint is reused across every target budget for that patient/seed.
- **Patient personalization:** RL starts from random weights; TL starts from the source checkpoint. Both use the exact same target data, normalization, chronological validation, batch ordering seed, learning rate, 50-epoch target cap and patience 10. Both reset Adam at the start of target training. All TL layers are trainable.
- **No temporal leakage:** preprocess each record once with forward-only event handling, then slice history budgets (the active pump setting can carry across the boundary); fit normalization on the fitting portion only; construct train and validation windows separately. Windows must have exact five-minute spacing and valid features. No interpolation across gaps. Each source has its own fitting-only normalization; RL and TL share the target's fitting-only normalization.
- **Fixed evaluation:** identical official test windows across budgets, modes and seeds; test data never select epochs or hyperparameters. Persistence is checked for invariance. Source records are treated as an already available historical library; their calendar dates are not restricted to the target date.
- **A separate full-history control:** rerun with this same protocol. Chronological validation, source-only checkpoint selection, gap checks and matched target-training caps differ from the published experiment. Comparisons to the article's numbers are therefore descriptive, not a clean isolation of history length. Compare budgets within this notebook.

Strict evaluation does not by itself prove why a result is small, and a cold-start gain cannot explain the change between two prior estimates. The manuscript's revised headline also reflects corrected calculations and changed saved configurations, not just added seeds. This notebook leaves the article unchanged.


In [ ]:
import os, sys, subprocess, json, shutil, hashlib, importlib.util
os.environ.setdefault('MPLCONFIGDIR', '/content/cold-start-matplotlib')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'numpy>=1.24,<3', 'pandas>=2,<3', 'scipy>=1.10,<2', 'matplotlib>=3.7,<4'], check=True)
import torch
if not torch.cuda.is_available() and not SMOKE_TEST:
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (smoke only)')


## 3. Embedded implementation

The runner is maintained in `RUN/experiments/run_cold_start.py`. The two preprocessing modules below are exact snapshots of the repository files; their hashes are recorded with every run. This notebook does not use `run_on_colab.ipynb`'s experiment folders.


In [ ]:
BUNDLE = {'run_cold_start.py': '#!/usr/bin/env python3\n"""History-budget transfer experiment. Isolated outputs; no manuscript edits."""\nfrom pathlib import Path\nimport contextlib\nimport copy\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport random\nimport shutil\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset, ConcatDataset\n\nCOHORT = {\'2018\': [559, 563, 570, 575, 588, 591],\n          \'2020\': [540, 544, 552, 567, 584, 596]}\nFEATURES = [\'glucose\', \'basal\', \'bolus\', \'carbs\']\nDEFAULTS = dict(model=\'gru\', horizon_steps=6, window=12, sampling_minutes=5,\n    hidden_size=64, layers=2, dropout=.2, batch_size=64,\n    source_epochs=10, target_epochs=50, source_patience=5, target_patience=10,\n    learning_rate=.001, validation_fraction=.2, budgets_days=[3, 5, 7, 10, \'full\'],\n    seeds=[41, 42, 43], patients=sorted(sum(COHORT.values(), [])),\n    min_train_windows=32, min_validation_windows=8, min_test_windows=32,\n    bootstrap_replicates=20000, analysis_seed=20260923, smoke=False)\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n    tmp.replace(path)\n\n\ndef load_support(folder):\n    modules = []\n    for name in [\'loaders\', \'preprocessors\']:\n        spec = importlib.util.spec_from_file_location(\'cold_\' + name, Path(folder) / f\'{name}.py\')\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        modules.append(module)\n    return modules\n\n\ndef clean_frame(raw, preprocessor):\n    # Called ONCE per record, BEFORE slicing, exactly as the published pipeline\n    # preprocesses a whole train/test file. Cleaning each slice separately\n    # instead looks stricter but is not: basic_preprocessing forward-fills basal\n    # and then drops any row still holding a NaN, so a slice containing no basal\n    # event loses every row. Basal events are sparse -- a 4.8 h validation tail\n    # usually holds none -- which emptied 10 of 12 patients at the 1-day budget.\n    # What crosses a budget boundary here is a pump setting carried forward in\n    # time, never a glucose observation: glucose is never filled (strategy\n    # \'none\'), and no fill moves backwards.\n    with contextlib.redirect_stdout(io.StringIO()):\n        df = preprocessor.preprocess_ohiot1dm_data(\n            {0: raw.copy()}, include_feature_engineering=False)[0]\n    if df.empty:\n        raise ValueError(\'Empty preprocessed segment\')\n    # A channel is absent when the patient logged no such event at all: patient\n    # 567\'s test record has no meals, so the loader emits no carbs column. The\n    # published pipeline substitutes zeros for an absent feature column\n    # (benchmark/data/torch_dataset.py), and the line below already zero-fills\n    # missing bolus/carbs values, so an absent event channel is zero here too.\n    # Glucose and basal are not event channels -- absence means a broken record.\n    for channel in [\'bolus\', \'carbs\']:\n        if channel not in df.columns:\n            df[channel] = 0.\n    absent = [c for c in FEATURES if c not in df.columns]\n    if absent:\n        raise ValueError(\'Missing required channel(s): \' + \', \'.join(absent))\n    df = df[FEATURES].apply(pd.to_numeric, errors=\'coerce\')\n    df.index = pd.to_datetime(df.index)\n    df = df.sort_index()\n    if df.index.has_duplicates:\n        raise ValueError(\'Duplicate preprocessed timestamps\')\n    df = df.replace(-1, np.nan)\n    df[[\'bolus\', \'carbs\']] = df[[\'bolus\', \'carbs\']].fillna(0.)\n    df.loc[~df.glucose.between(20, 600), \'glucose\'] = np.nan\n    return df\n\n\ndef cgm_bounds(raw):\n    glucose = pd.to_numeric(raw.glucose, errors=\'coerce\')\n    times = pd.DatetimeIndex(raw.index)[glucose.between(20, 600)]\n    if len(times) < 2:\n        raise ValueError(\'Insufficient glucose history\')\n    return times.min(), times.max()\n\n\ndef budget_split(frame, days, cfg):\n    # `frame` is already preprocessed; slicing it keeps every budget on the same\n    # cleaned rows, so budgets differ only in how much history they include.\n    raw = frame\n    start, last = cgm_bounds(raw)\n    end = last + pd.Timedelta(minutes=cfg[\'sampling_minutes\'])\n    if days != \'full\':\n        requested = end - pd.Timedelta(days=float(days))\n        if requested < start:\n            raise ValueError(f\'History shorter than requested {days} days\')\n        start = requested\n    # All budgets end together to avoid conflating data volume with recency.\n    selected = raw.loc[(raw.index >= start) & (raw.index < end)].copy()\n    boundary = start + (end - start) * (1 - cfg[\'validation_fraction\'])\n    train = selected.loc[selected.index < boundary].copy()\n    val = selected.loc[selected.index >= boundary].copy()\n    if train.empty or val.empty:\n        raise ValueError(\'Empty chronological training or validation segment\')\n    return train, val, dict(history_start=str(start), history_end_exclusive=str(end),\n                           validation_start=str(boundary), budget_days=days)\n\n\ndef fit_scaler(frame):\n    values = frame.to_numpy(float)\n    mean, sd = np.nanmean(values, axis=0), np.nanstd(values, axis=0)\n    if not np.isfinite(mean).all() or not np.isfinite(sd).all():\n        raise ValueError(\'Feature has no valid training observations\')\n    return mean, np.where(sd > 1e-8, sd, 1.)\n\n\ndef windows(frame, cfg, mean, sd):\n    a = frame.to_numpy(np.float32)\n    timestamps = frame.index.to_numpy(dtype=\'datetime64[ns]\')\n    w, h = cfg[\'window\'], cfg[\'horizon_steps\']\n    length = w + h\n    if len(a) < length:\n        return None\n    valid = np.isfinite(a).all(axis=1)\n    # Prefix sums test every adjacent interval AND all rows in the window.\n    invalid = np.r_[0, np.cumsum(~valid)]\n    gap = np.r_[0, np.diff(timestamps) != np.timedelta64(cfg[\'sampling_minutes\'], \'m\')]\n    gaps = np.r_[0, np.cumsum(gap)]\n    starts = np.arange(len(a) - length + 1)\n    good = ((invalid[starts+length] - invalid[starts]) == 0) & (\n        (gaps[starts+length] - gaps[starts+1]) == 0)\n    starts = starts[good]\n    if not len(starts):\n        return None\n    z = ((a - mean) / sd).astype(np.float32)\n    x = np.stack([z[i:i+w] for i in starts])\n    y = np.stack([z[i+w:i+length, 0] for i in starts])\n    return dict(dataset=TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),\n                timestamps=timestamps[starts+length-1],\n                truth=a[starts+length-1, 0].astype(float),\n                persistence=a[starts+w-1, 0].astype(float))\n\n\ndef prepare_segments(clean_train, clean_test, days, cfg):\n    train, val, audit = budget_split(clean_train, days, cfg)\n    if train.empty or val.empty:\n        raise ValueError(\'Empty preprocessed segment\')\n    mean, sd = fit_scaler(train)\n    train_w, val_w = windows(train, cfg, mean, sd), windows(val, cfg, mean, sd)\n    # Report the counts: a budget that is short by one window and a budget with\n    # a sensor outage covering the whole slice need different answers.\n    n_train_w = len(train_w[\'dataset\']) if train_w else 0\n    n_val_w = len(val_w[\'dataset\']) if val_w else 0\n    if n_train_w < cfg[\'min_train_windows\']:\n        raise ValueError(f"Too few contiguous training windows: {n_train_w} < "\n                         f"{cfg[\'min_train_windows\']} (budget {days})")\n    if n_val_w < cfg[\'min_validation_windows\']:\n        raise ValueError(f"Too few contiguous validation windows: {n_val_w} < "\n                         f"{cfg[\'min_validation_windows\']} (budget {days})")\n    audit.update(n_train=len(train_w[\'dataset\']), n_validation=len(val_w[\'dataset\']),\n                 normalization_mean=mean.tolist(), normalization_sd=sd.tolist())\n    result = dict(train=train_w, val=val_w, mean=mean, sd=sd, audit=audit)\n    if clean_test is not None:\n        test = clean_test\n        if train.index.max() >= val.index.min() or val.index.max() >= test.index.min():\n            raise ValueError(\'Train/validation/test chronology violated\')\n        test_w = windows(test, cfg, mean, sd)\n        n_test_w = len(test_w[\'dataset\']) if test_w else 0\n        if n_test_w < cfg[\'min_test_windows\']:\n            raise ValueError(f"Too few contiguous test windows: {n_test_w} < "\n                             f"{cfg[\'min_test_windows\']}")\n        audit.update(n_test=len(test_w[\'dataset\']), test_start=str(test.index.min()),\n                     test_end=str(test.index.max()))\n        result[\'test\'] = test_w\n    return result\n\n\nclass ForecastNet(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        cls = {\'gru\': nn.GRU, \'lstm\': nn.LSTM, \'rnn\': nn.RNN}[cfg[\'model\']]\n        self.encoder = cls(4, cfg[\'hidden_size\'], cfg[\'layers\'], batch_first=True,\n                          dropout=cfg[\'dropout\'] if cfg[\'layers\'] > 1 else 0.)\n        self.head = nn.Linear(cfg[\'hidden_size\'], cfg[\'horizon_steps\'])\n\n    def forward(self, x):\n        sequence, _ = self.encoder(x)\n        return self.head(sequence[:, -1])\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef fit(model, training, validation, cfg, device, seed, epochs, patience):\n    seed_everything(seed)\n    generator = torch.Generator().manual_seed(seed)\n    train = DataLoader(training, batch_size=cfg[\'batch_size\'], shuffle=True,\n                       generator=generator, pin_memory=device.startswith(\'cuda\'))\n    val = DataLoader(validation, batch_size=cfg[\'batch_size\'], shuffle=False)\n    model.to(device)\n    # Always fresh, including fine-tuning: no optimizer state crosses stages.\n    optimizer = torch.optim.Adam(model.parameters(), lr=cfg[\'learning_rate\'])\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=.5, patience=1)\n    best, stale, state = float(\'inf\'), 0, None\n    history = []\n    for epoch in range(epochs):\n        model.train()\n        total, count = 0., 0\n        for x, y in train:\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = nn.functional.mse_loss(model(x), y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite training loss\')\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            optimizer.step()\n            total += loss.item() * len(y)\n            count += len(y)\n        model.eval()\n        vsum, vn = 0., 0\n        with torch.no_grad():\n            for x, y in val:\n                pred = model(x.to(device))\n                loss = nn.functional.mse_loss(pred, y.to(device))\n                vsum += loss.item() * len(y)\n                vn += len(y)\n        vl = vsum / vn\n        if not np.isfinite(vl):\n            raise ValueError(\'Nonfinite validation loss\')\n        scheduler.step(vl)\n        history.append(dict(epoch=epoch+1, train_mse=total/count, validation_mse=vl))\n        print(f\'    epoch {epoch+1}/{epochs}: val MSE {vl:.5f}\', flush=True)\n        if vl < best:\n            best, stale = vl, 0\n            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        if stale >= patience:\n            break\n    model.load_state_dict(state)\n    return history\n\n\ndef evaluate(model, pack, cfg, device):\n    model.eval()\n    pred = []\n    with torch.no_grad():\n        for x, _ in DataLoader(pack[\'test\'][\'dataset\'], batch_size=cfg[\'batch_size\']):\n            pred.extend(model(x.to(device))[:, -1].cpu().numpy())\n    pred = np.array(pred)*pack[\'sd\'][0] + pack[\'mean\'][0]\n    truth = pack[\'test\'][\'truth\']\n    diff = pred - truth\n    persistence = pack[\'test\'][\'persistence\']\n    metrics = dict(mae=float(abs(diff).mean()), rmse=float(np.sqrt(np.mean(diff**2))),\n                   persistence_mae=float(abs(persistence-truth).mean()),\n                   persistence_rmse=float(np.sqrt(np.mean((persistence-truth)**2))), n_test=len(truth))\n    table = pd.DataFrame(dict(target_timestamp=pack[\'test\'][\'timestamps\'], truth=truth,\n                             prediction=pred, persistence=persistence))\n    return metrics, table\n\n\ndef make_protocol(data_root, support, cfg):\n    files = []\n    for release, patients in COHORT.items():\n        for pid in patients:\n            if pid not in cfg[\'patients\']:\n                continue\n            for mode in [\'train\', \'test\']:\n                p = Path(data_root)/\'raw\'/\'ohiot1dm\'/release/mode/f\'{pid}-ws-{mode}ing.xml\'\n                if not p.is_file():\n                    raise FileNotFoundError(p)\n                files.append(dict(path=str(p.relative_to(data_root)), sha256=digest(p)))\n    return dict(config=cfg, input_files=files, torch_version=torch.__version__,\n                numpy_version=np.__version__, pandas_version=pd.__version__,\n                sources={name: digest(Path(support)/name) for name in [\'loaders.py\', \'preprocessors.py\']},\n                runner_sha256=digest(__file__),\n                design=\'most recent target-history budget; chronological validation; fixed official test; \'\n                       \'leave-target-out sources; preprocessing applied once per record before slicing\')\n\n\ndef run(data_root, support, cfg, local_root, drive_root, device=\'cuda\'):\n    cfg = copy.deepcopy(cfg)\n    if device.startswith(\'cuda\') and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training.\')\n    if cfg[\'model\'] not in [\'gru\', \'lstm\', \'rnn\'] or cfg[\'sampling_minutes\'] != 5:\n        raise ValueError(\'Supported models: GRU/LSTM/RNN; this protocol uses five-minute data.\')\n    if len(cfg[\'patients\']) < 2 or len(set(cfg[\'patients\'])) != len(cfg[\'patients\']):\n        raise ValueError(\'Need distinct target/source patients\')\n    if \'full\' not in cfg[\'budgets_days\'] or len(set(cfg[\'budgets_days\'])) != len(cfg[\'budgets_days\']):\n        raise ValueError(\'Include one full-history control and distinct budgets\')\n    protocol = make_protocol(data_root, support, cfg)\n    fingerprint = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]\n    name = (\'SMOKE_\' if cfg[\'smoke\'] else \'\') + f"{cfg[\'model\']}_{5*cfg[\'horizon_steps\']}min_{fingerprint}"\n    local, remote = Path(local_root)/name, Path(drive_root)/name\n    local.mkdir(parents=True, exist_ok=True)\n    remote.mkdir(parents=True, exist_ok=True)\n    # Restore only this exact data/config/code fingerprint. No stale result reuse.\n    shutil.copytree(remote, local, dirs_exist_ok=True)\n    atomic_json(local/\'protocol.json\', protocol)\n    shutil.copy2(local/\'protocol.json\', remote/\'protocol.json\')\n    loader, prep = load_support(support)\n    raw = {}\n    for pid in cfg[\'patients\']:\n        release = next(v for v, ids in COHORT.items() if pid in ids)\n        raw[pid] = {}\n        for mode in [\'train\', \'test\']:\n            with contextlib.redirect_stdout(io.StringIO()):\n                raw[pid][mode] = loader.load_ohiot1dm_data(\n                    str(data_root), patient_ids=[pid], mode=mode, version=release)[pid]\n            raw[pid][mode].index = pd.to_datetime(raw[pid][mode].index)\n    # Preflight EVERY budget before spending GPU time. Eligibility cannot be\n    # selected by observed performance. Detailed errors are saved, then fail.\n    source, audits, errors, clean = {}, [], [], {}\n    print(\'Preflighting every patient/budget and source split...\', flush=True)\n    for pid in cfg[\'patients\']:\n        try:\n            # Preprocess each record once; every budget then slices these rows.\n            clean[pid] = {mode: clean_frame(raw[pid][mode], prep) for mode in [\'train\', \'test\']}\n            source[pid] = prepare_segments(clean[pid][\'train\'], None, \'full\', cfg)\n            reference = None\n            for budget in cfg[\'budgets_days\']:\n                pack = prepare_segments(clean[pid][\'train\'], clean[pid][\'test\'], budget, cfg)\n                current = pack[\'test\'][\'timestamps\']\n                if reference is not None and not np.array_equal(current, reference):\n                    raise ValueError(\'Test windows change across budgets\')\n                reference = current\n                audits.append(dict(patient_id=pid, **pack[\'audit\']))\n        except Exception as exc:\n            errors.append(dict(patient_id=pid, error=str(exc)))\n    atomic_json(local/\'preflight.json\', dict(audits=audits, errors=errors))\n    shutil.copy2(local/\'preflight.json\', remote/\'preflight.json\')\n    if errors:\n        raise ValueError(f\'Preflight failed; inspect {remote}/preflight.json. No model trained. {errors}\')\n    print(f\'Preflight passed. Outputs: {remote}\', flush=True)\n    for seed in cfg[\'seeds\']:\n        for pid in cfg[\'patients\']:\n            source_dir = local/\'pretraining\'/f\'patient{pid}_seed{seed}\'\n            source_remote = remote/\'pretraining\'/source_dir.name\n            source_dir.mkdir(parents=True, exist_ok=True)\n            checkpoint = source_dir/\'weights.pt\'\n            complete_source = source_dir/\'complete.json\'\n            valid_source = (complete_source.is_file() and checkpoint.is_file() and\n                            json.loads(complete_source.read_text()).get(\'checkpoint_sha256\') == digest(checkpoint))\n            if not valid_source:\n                print(f\'Pretrain excluding patient {pid}, seed {seed}\', flush=True)\n                seed_everything(seed)\n                model = ForecastNet(cfg)\n                donors = [q for q in cfg[\'patients\'] if q != pid]\n                history = fit(model, ConcatDataset([source[q][\'train\'][\'dataset\'] for q in donors]),\n                              ConcatDataset([source[q][\'val\'][\'dataset\'] for q in donors]),\n                              cfg, device, seed, cfg[\'source_epochs\'], cfg[\'source_patience\'])\n                torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, checkpoint)\n                atomic_json(complete_source, dict(donors=donors, target_excluded=pid, history=history,\n                                                  checkpoint_sha256=digest(checkpoint)))\n                source_remote.mkdir(parents=True, exist_ok=True)\n                shutil.copy2(checkpoint, source_remote/\'weights.pt\')\n                shutil.copy2(complete_source, source_remote/\'complete.json\')\n                del model\n            expected_hash = json.loads(complete_source.read_text())[\'checkpoint_sha256\']\n            if digest(checkpoint) != expected_hash:\n                raise ValueError(\'Cached source checkpoint hash mismatch\')\n            for budget in cfg[\'budgets_days\']:\n                job = local/\'jobs\'/f\'patient{pid}_seed{seed}_days{budget}\'\n                target = remote/\'jobs\'/job.name\n                done = job/\'complete.json\'\n                if done.is_file():\n                    saved = json.loads(done.read_text())\n                    hashes = saved.get(\'files_sha256\', {})\n                    if hashes and all((job/name).is_file() and digest(job/name) == value\n                                      for name, value in hashes.items()):\n                        target.mkdir(parents=True, exist_ok=True)\n                        for name in hashes:\n                            shutil.copy2(job/name, target/name)\n                        shutil.copy2(done, target/\'complete.json\')\n                        print(f\'Skip completed {job.name}\', flush=True)\n                        continue\n                job.mkdir(parents=True, exist_ok=True)\n                pack = prepare_segments(clean[pid][\'train\'], clean[pid][\'test\'], budget, cfg)\n                row = dict(patient_id=pid, seed=seed, **pack[\'audit\'])\n                start_time = time.perf_counter()\n                for mode in [\'regular\', \'transfer\']:\n                    print(f\'{job.name}: {mode}\', flush=True)\n                    seed_everything(seed)\n                    model = ForecastNet(cfg)\n                    if mode == \'transfer\':\n                        model.load_state_dict(torch.load(checkpoint, map_location=\'cpu\', weights_only=True))\n                    history = fit(model, pack[\'train\'][\'dataset\'], pack[\'val\'][\'dataset\'], cfg,\n                                  device, seed, cfg[\'target_epochs\'], cfg[\'target_patience\'])\n                    metrics, predictions = evaluate(model, pack, cfg, device)\n                    row[mode] = metrics\n                    atomic_json(job/f\'{mode}_history.json\', history)\n                    predictions.to_csv(job/f\'{mode}_predictions.csv\', index=False)\n                    del model\n                row[\'seconds\'] = time.perf_counter() - start_time\n                row[\'files_sha256\'] = {p.name: digest(p) for p in job.iterdir()\n                                      if p.name.endswith((\'_history.json\', \'_predictions.csv\'))}\n                # Remote completion marker is copied LAST: interruption cannot\n                # make an incomplete remote job look complete on next runtime.\n                target.mkdir(parents=True, exist_ok=True)\n                for path in job.iterdir():\n                    if path.name != \'complete.json\':\n                        shutil.copy2(path, target/path.name)\n                atomic_json(done, row)\n                shutil.copy2(done, target/\'complete.json\')\n                print(f\'Completed and mirrored {job.name}: {row["seconds"]/60:.1f} min\', flush=True)\n    summarize(local, remote, cfg)\n    return local, remote\n\n\ndef summarize(local, remote, cfg):\n    from scipy.stats import wilcoxon\n    rows = [json.loads(p.read_text()) for p in sorted((Path(local)/\'jobs\').glob(\'*/complete.json\'))]\n    expected = {(p, s, str(b)) for p in cfg[\'patients\'] for s in cfg[\'seeds\'] for b in cfg[\'budgets_days\']}\n    got = {(r[\'patient_id\'], r[\'seed\'], str(r[\'budget_days\'])) for r in rows}\n    if got != expected or len(rows) != len(expected):\n        raise ValueError(\'Incomplete or duplicate cohort: summary withheld until all paired jobs finish\')\n    flat = []\n    for r in rows:\n        flat.append(dict(patient_id=r[\'patient_id\'], seed=r[\'seed\'], budget_days=str(r[\'budget_days\']),\n                         n_train=r[\'n_train\'], n_validation=r[\'n_validation\'], n_test=r[\'n_test\'],\n                         **{f\'{mode}_{metric}\': r[mode][metric] for mode in [\'regular\', \'transfer\']\n                            for metric in [\'mae\', \'rmse\', \'persistence_mae\', \'persistence_rmse\']}))\n    frame = pd.DataFrame(flat)\n    np.testing.assert_allclose(frame.regular_persistence_mae, frame.transfer_persistence_mae)\n    if (frame.groupby(\'patient_id\').regular_persistence_mae.nunique() != 1).any():\n        raise ValueError(\'Persistence/test values differ across budgets/seeds\')\n    out = Path(local)/\'summary\'\n    out.mkdir(exist_ok=True)\n    frame.to_csv(out/\'patient_seed_metrics.csv\', index=False)\n    rng = np.random.default_rng(cfg[\'analysis_seed\'])\n    n, k = len(cfg[\'patients\']), len(cfg[\'seeds\'])\n    reps = cfg[\'bootstrap_replicates\']\n    pi = rng.integers(0, n, size=(reps, n))\n    si = rng.integers(0, k, size=(reps, k))\n    summaries = []\n    def array(budget, mode, metric):\n        return frame[frame.budget_days == str(budget)].pivot(index=\'patient_id\', columns=\'seed\',\n            values=f\'{mode}_{metric}\').loc[cfg[\'patients\'], cfg[\'seeds\']].to_numpy()\n    for metric in [\'mae\', \'rmse\']:\n        full = array(\'full\', \'regular\', metric) - array(\'full\', \'transfer\', metric)\n        for budget in cfg[\'budgets_days\']:\n            rl, tl = array(budget, \'regular\', metric), array(budget, \'transfer\', metric)\n            diff = rl-tl\n            sampled_rl = rl[pi[:, :, None], si[:, None, :]].mean(axis=(1, 2))\n            sampled_diff = diff[pi[:, :, None], si[:, None, :]].mean(axis=(1, 2))\n            full_diff = full[pi[:, :, None], si[:, None, :]].mean(axis=(1, 2))\n            percent = 100*sampled_diff/sampled_rl\n            per_patient = diff.mean(axis=1)\n            p = 1. if np.allclose(per_patient, 0) else float(wilcoxon(per_patient).pvalue)\n            summaries.append(dict(metric=metric, budget_days=str(budget),\n                regular_mean=float(rl.mean()), transfer_mean=float(tl.mean()),\n                benefit=float(diff.mean()), benefit_ci_low=float(np.quantile(sampled_diff,.025)),\n                benefit_ci_high=float(np.quantile(sampled_diff,.975)),\n                relative_benefit_pct=float(100*diff.mean()/rl.mean()),\n                relative_ci_low=float(np.quantile(percent,.025)), relative_ci_high=float(np.quantile(percent,.975)),\n                additional_benefit_vs_full=float((diff-full).mean()),\n                additional_ci_low=float(np.quantile(sampled_diff-full_diff,.025)),\n                additional_ci_high=float(np.quantile(sampled_diff-full_diff,.975)),\n                wilcoxon_p=p, n_patients=n, n_seeds=k))\n    table = pd.DataFrame(summaries)\n    ps = table.wilcoxon_p.to_numpy(); order = np.argsort(ps); q = np.empty(len(ps))\n    q[order] = np.minimum.accumulate((ps[order]*len(ps)/np.arange(1,len(ps)+1))[::-1])[::-1]\n    table[\'wilcoxon_bh_q\'] = np.minimum(q,1)\n    table.to_csv(out/\'budget_summary.csv\', index=False)\n    text = [\'# Cold-start history-budget experiment\', \'\',\n        \'SMOKE RUN — not article evidence.\' if cfg[\'smoke\'] else \'Exploratory follow-up; no outcome-driven budget selection.\',\n        \'\', f"Model: {cfg[\'model\'].upper()}, horizon: {5*cfg[\'horizon_steps\']} minutes; {n} patients, {k} seeds.",\n        \'Budgets use the most recent available training history, including the chronological validation portion. Official test windows are fixed across all budgets.\',\n        \'Positive benefit = RL error minus TL error. Percent benefit is the reduction in equal-patient cohort mean error, not a pooled-window improvement.\',\n        \'95% intervals jointly resample whole patients and shared training seeds; additional-benefit intervals use the same draws for the budget and full-history control. Intervals are pointwise, not simultaneous.\',\n        \'Wilcoxon tests use patient cross-seed means; BH spans both metrics and every budget in this run. Different architectures/horizons run separately would require a broader family before combined claims.\',\n        \'\', \'## Interpretation\', \'\',\n        \'The primary comparison is MAE at 3 days versus this experiment\\\'s full-history control. A larger advantage is a hypothesis, not guaranteed. Report all budgets, including null or adverse effects. Inspect absolute RL/TL errors and persistence as well as percentage gains.\',\n        \'This is simulated limited-history personalization with a retrospective source library, not a prospectively recruited incident-patient study. Other patients\\\' source records are assumed available; their calendar dates are not restricted to each target\\\'s test date.\',\n        \'Chronological validation, gap-checked windows, source-only pretraining validation, and matched 50-epoch target caps differ from the article\\\'s original schedule. Compare budgets within this new experiment; do not replace the published full-history headline with its best budget.\',\n        \'The budgets share the same final training time, controlling history recency. Test separation/gaps follow the official Ohio split. This experiment cannot explain why earlier submitted numbers changed.\',\n        \'\', \'See budget_summary.csv, patient_seed_metrics.csv, preflight.json and protocol.json. Per-job predictions and training histories are retained.\', \'\']\n    (out/\'REPORT.md\').write_text(\'\\n\'.join(text))\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig, axes = plt.subplots(1,2,figsize=(10,4),constrained_layout=True)\n    for ax,metric in zip(axes,[\'mae\',\'rmse\']):\n        t = table[table.metric == metric].set_index(\'budget_days\').loc[list(map(str,cfg[\'budgets_days\']))]\n        x = np.arange(len(t))\n        ax.plot(x,t.relative_benefit_pct,marker=\'o\')\n        ax.vlines(x,t.relative_ci_low,t.relative_ci_high)\n        ax.axhline(0,color=\'gray\',lw=.8)\n        ax.set(xticks=x,xticklabels=t.index,xlabel=\'Target history (days; full = all)\',\n               ylabel=f\'{metric.upper()} reduction (%)\',title=f\'{metric.upper()}: joint patient/seed 95% intervals\')\n    fig.savefig(out/\'history_budget_curve.png\',dpi=180)\n    plt.close(fig)\n    shutil.copytree(out,Path(remote)/\'summary\',dirs_exist_ok=True)\n    print(table.to_string(index=False),flush=True)\n', 'loaders.py': '"""\nData loaders\n\nThis module provides standardized data loading functions for the Ohio T1DM blood glucose datasets commonly used in research. The loaders return\nraw combined dataframes without any data alterations.\n\nAll data preprocessing and alterations should be done using the \npreprocessors module.\n\nSupported dataset:\n- OhioT1DM Dataset (2018 and 2020 versions)\n\nFor other datasets, please refer to the documentation for details on loading and preprocessing steps. \n"""\n\nimport os\nimport xml.etree.ElementTree as ET\nimport pandas as pd # type: ignore\nimport numpy as np\nimport datetime\nfrom typing import List, Dict, Optional, Tuple\n\n\nclass OhioT1DMDataLoader:\n    """\n    Data loader for the OhioT1DM dataset.\n    \n    This loader handles XML files from the OhioT1DM dataset and converts them\n    into raw structured pandas DataFrames with proper time series alignment.\n    No data alterations or preprocessing is applied - the data is returned\n    as-is from the XML files after basic parsing and merging.\n    \n    All data processing should be done using the OhioBGDataPreprocessor class.\n    \n    Attributes:\n        data_dir (str): Path to the data directory\n        sampling_rate (int): Target sampling rate in minutes (default: 5)\n        version (str): Dataset version (\'2018\' or \'2020\')\n    """\n\n    def __init__(self, data_dir: str, sampling_rate: int = 5, version: List[str] = [\'2018\', \'2020\']):\n        """\n        Initialize the OhioT1DM data loader.\n        \n        Args:\n            data_dir: Path to the data directory containing XML files\n            sampling_rate: Target sampling rate in minutes\n            version: Dataset version (\'2018\' or \'2020\')\n        """\n        self.data_dir = data_dir\n        self.sampling_rate = sampling_rate\n        self.version = version\n        \n        # Patient IDs for different versions\n        self.patient_ids = {\n            \'2018\': [559, 563, 570, 575, 588, 591],\n            \'2020\': [540, 544, 552, 567, 584, 596]\n        }\n    \n    def round_minute(self, date_string: str, round2min: int = 5):\n        """\n        Round datetime to specified minute intervals.\n        \n        Args:\n            date_string: Date string in format "%d-%m-%Y %H:%M:%S"\n            round2min: Minutes to round to\n            \n        Returns:\n            Rounded datetime object\n        """\n        date = datetime.datetime.strptime(date_string, "%d-%m-%Y %H:%M:%S")\n        new_min = ((date.minute // round2min) * round2min) # Round down to nearest interval E.g. 4 -> 0, 6->5 , 12 -> 10\n        date = date.replace(minute=int(new_min), second=0)\n        return date\n    \n    def get_cgm(self, root: ET.Element):\n        """Extract CGM glucose data from XML."""\n        glucose = []\n        glucose_ts = []\n        for event in root.findall(\'glucose_level/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            glucose.append(value)\n            glucose_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': glucose_ts,\n            \'glucose\': glucose\n        }).set_index(\'ts\')\n    \n    def get_fingerstick(self, root: ET.Element):\n        """Extract fingerstick glucose data from XML."""\n        fingerstick = []\n        fingerstick_ts = []\n        for event in root.findall(\'finger_stick/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            fingerstick.append(value)\n            fingerstick_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': fingerstick_ts,\n            \'fingerstick\': fingerstick\n        }).set_index(\'ts\')\n    \n    def get_gsr(self, root: ET.Element):\n        """Extract galvanic skin response data from XML."""\n        gsr = []\n        gsr_ts = []\n        for event in root.findall(\'basis_gsr/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            gsr.append(value)\n            gsr_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': gsr_ts,\n            \'gsr\': gsr\n        }).set_index(\'ts\')\n    \n    def get_heart_rate(self, root: ET.Element):\n        """Extract heart rate data from XML."""\n        hr = []\n        hr_ts = []\n        for event in root.findall(\'basis_heart_rate/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            hr.append(value)\n            hr_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': hr_ts,\n            \'hr\': hr\n        }).set_index(\'ts\')\n    \n    def get_skin_temperature(self, root: ET.Element):\n        """Extract skin temperature data from XML."""\n        st = []\n        st_ts = []\n        for event in root.findall(\'basis_skin_temperature/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            st.append(value)\n            st_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': st_ts,\n            \'st\': st\n        }).set_index(\'ts\')\n    \n    def get_basal(self, root: ET.Element):\n        """Extract basal insulin data from XML."""\n        basal = []\n        basal_ts = []\n        for event in root.findall(\'basal/event\'):\n            value = event.get(\'value\')\n            ts = event.get(\'ts\')\n            ts = self.round_minute(ts, self.sampling_rate)\n            basal.append(value)\n            basal_ts.append(ts)\n        \n        return pd.DataFrame({\n            \'ts\': basal_ts,\n            \'basal\': basal\n        }).set_index(\'ts\')\n    \n    def get_temp_basal(self, root: ET.Element):\n        """Extract temporary basal insulin data from XML."""\n        temp_basal = []\n        temp_basal_ts = []\n        basal_end = []\n        \n        for event in root.findall(\'temp_basal/event\'):\n            value = event.get(\'value\')\n            ts_begin = event.get(\'ts_begin\')\n            ts_end = event.get(\'ts_end\')\n            \n            ts_begin = self.round_minute(ts_begin, self.sampling_rate)\n            ts_end = self.round_minute(ts_end, self.sampling_rate)\n            \n            temp_basal.append(value)\n            temp_basal_ts.append(ts_begin)\n            basal_end.append(ts_end)\n        \n        return pd.DataFrame({\n            \'ts\': temp_basal_ts,\n            \'temp_basal\': temp_basal,\n            \'basal_end\': basal_end\n        }).set_index(\'ts\')\n    \n    def get_bolus(self, root: ET.Element):\n        """Extract bolus insulin data from XML."""\n        bolus = []\n        bolus_ts = []\n        bolus_end = []\n        bolus_dur = []\n        \n        for event in root.findall(\'bolus/event\'):\n            dose = event.get(\'dose\')\n            ts_begin = event.get(\'ts_begin\')\n            ts_end = event.get(\'ts_end\')\n            \n            ts_begin = self.round_minute(ts_begin, self.sampling_rate)\n            ts_end = self.round_minute(ts_end, self.sampling_rate)\n            \n            duration = (ts_end - ts_begin).seconds // 60\n            \n            bolus.append(dose)\n            bolus_ts.append(ts_begin)\n            bolus_end.append(ts_end)\n            bolus_dur.append(duration)\n        \n        return pd.DataFrame({\n            \'ts\': bolus_ts,\n            \'bolus\': bolus,\n            \'bolus_dur\': bolus_dur,\n            \'bolus_end\': bolus_end\n        }).set_index(\'ts\')\n    \n    def get_meal(self, root: ET.Element):\n        """Extract meal/carbohydrate data from XML."""\n        carbs = []\n        meal_ts = []\n        meal_type = []\n        \n        for event in root.findall(\'meal/event\'):\n            carb_value = event.get(\'carbs\')\n            ts = event.get(\'ts\')\n            meal_type_value = event.get(\'type\')\n            \n            ts = self.round_minute(ts, self.sampling_rate)\n            \n            carbs.append(carb_value)\n            meal_ts.append(ts)\n            meal_type.append(meal_type_value)\n        \n        return pd.DataFrame({\n            \'ts\': meal_ts,\n            \'carbs\': carbs,\n            \'meal_type\': meal_type\n        }).set_index(\'ts\')\n    \n    def get_exercise(self, root: ET.Element):\n        """Extract exercise data from XML."""\n        exercise_intensity = []\n        exercise_ts = []\n        exercise_dur = []\n        \n        for event in root.findall(\'exercise/event\'):\n            intensity = event.get(\'intensity\')\n            ts = event.get(\'ts\')\n            duration = event.get(\'duration\')\n            \n            ts = self.round_minute(ts, self.sampling_rate)\n            \n            exercise_intensity.append(intensity)\n            exercise_ts.append(ts)\n            exercise_dur.append(duration)\n        \n        return pd.DataFrame({\n            \'ts\': exercise_ts,\n            \'exercise_intensity\': exercise_intensity,\n            \'exer_dur\': exercise_dur\n        }).set_index(\'ts\')\n    \n    def get_sleep(self, root: ET.Element):\n        """Extract sleep data from XML."""\n        sleep_quality = []\n        sleep_ts = []\n        sleep_dur = []\n        sleep_end = []\n        \n        for event in root.findall(\'sleep/event\'):\n            quality = event.get(\'quality\')\n            # Note: ts_end and ts_begin are swapped in the dataset\n            ts_begin = event.get(\'ts_end\')\n            ts_end = event.get(\'ts_begin\')\n            \n            ts_begin = self.round_minute(ts_begin, self.sampling_rate)\n            ts_end = self.round_minute(ts_end, self.sampling_rate)\n            \n            duration = (ts_end - ts_begin).seconds // 60\n            \n            sleep_quality.append(quality)\n            sleep_ts.append(ts_begin)\n            sleep_dur.append(duration)\n            sleep_end.append(ts_end)\n        \n        return pd.DataFrame({\n            \'ts\': sleep_ts,\n            \'sleep\': sleep_quality,\n            \'sleep_dur\': sleep_dur,\n            \'sleep_end\': sleep_end\n        }).set_index(\'ts\')\n    \n    def get_work(self, root: ET.Element):\n        """Extract work stress data from XML."""\n        work_intensity = []\n        work_ts = []\n        work_dur = []\n        work_end = []\n        \n        for event in root.findall(\'work/event\'):\n            intensity = event.get(\'intensity\')\n            ts_begin = event.get(\'ts_begin\')\n            ts_end = event.get(\'ts_end\')\n            \n            ts_begin = self.round_minute(ts_begin, self.sampling_rate)\n            ts_end = self.round_minute(ts_end, self.sampling_rate)\n            \n            duration = (ts_end - ts_begin).seconds // 60\n            \n            work_intensity.append(intensity)\n            work_ts.append(ts_begin)\n            work_dur.append(duration)\n            work_end.append(ts_end)\n        \n        return pd.DataFrame({\n            \'ts\': work_ts,\n            \'work\': work_intensity,\n            \'work_dur\': work_dur,\n            \'work_end\': work_end\n        }).set_index(\'ts\')\n\n    def load_single_file(self, file_path: str):\n        """\n        Load and parse a single XML file from the OhioT1DM dataset.\n        \n        Args:\n            file_path: Path to the XML file\n            \n        Returns:\n            Merged DataFrame with all data types\n        """\n        root = ET.parse(file_path).getroot()\n        \n        # Extract all data types\n        dataframes = []\n        \n        # Core continuous glucose data\n        cgm_df = self.get_cgm(root)\n        if not cgm_df.empty:\n            dataframes.append(cgm_df)\n        \n        # Wearable sensor data (GSR, HR, ST)\n        gsr_df = self.get_gsr(root)\n        if not gsr_df.empty:\n            dataframes.append(gsr_df)\n            \n        hr_df = self.get_heart_rate(root)\n        if not hr_df.empty:\n            dataframes.append(hr_df)\n            \n        st_df = self.get_skin_temperature(root)\n        if not st_df.empty:\n            dataframes.append(st_df)\n        \n        # Insulin data\n        basal_df = self.get_basal(root)\n        if not basal_df.empty:\n            dataframes.append(basal_df)\n            \n        temp_basal_df = self.get_temp_basal(root)\n        if not temp_basal_df.empty:\n            dataframes.append(temp_basal_df)\n            \n        bolus_df = self.get_bolus(root)\n        if not bolus_df.empty:\n            dataframes.append(bolus_df)\n        \n        # Lifestyle data\n        meal_df = self.get_meal(root)\n        if not meal_df.empty:\n            dataframes.append(meal_df)\n            \n        exercise_df = self.get_exercise(root)\n        if not exercise_df.empty:\n            dataframes.append(exercise_df)\n            \n        sleep_df = self.get_sleep(root)\n        if not sleep_df.empty:\n            dataframes.append(sleep_df)\n            \n        work_df = self.get_work(root)\n        if not work_df.empty:\n            dataframes.append(work_df)\n        \n        # Merge all dataframes\n        if not dataframes:\n            return pd.DataFrame()\n        \n        merged_df = dataframes[0]\n        for df in dataframes[1:]:\n            merged_df = merged_df.join(df, how="outer")\n        \n        return merged_df\n    \n    def load_patient_data(self, patient_id: int, mode: str = \'train\'):\n        """\n        Load raw data for a specific patient.\n        \n        Args:\n            patient_id: Patient identifier\n            mode: \'train\' or \'test\'\n            \n        Returns:\n            Raw merged DataFrame for the patient (no processing applied)\n        """\n        # Determine which version this patient belongs to\n        patient_version = None\n        for version in self.version if isinstance(self.version, list) else [self.version]:\n            if patient_id in self.patient_ids[version]:\n                patient_version = version\n                break\n\n        if patient_version is None:\n            raise ValueError(f"Patient {patient_id} not found in any of the specified versions: {self.version}")\n\n        # Use the determined version for file path construction\n        file_pattern = f"{patient_id}-ws-{mode}ing.xml"\n        file_path = os.path.join(\n            self.data_dir, \n            \'raw\', \n            \'ohiot1dm\', \n            patient_version, \n            mode, \n            file_pattern\n        )\n        \n        if not os.path.exists(file_path):\n            raise FileNotFoundError(f"Data file not found: {file_path}")\n        \n        print(f"Loading data for patient {patient_id} ({mode} set)")\n        df = self.load_single_file(file_path)\n        \n        return df\n    \n    def load_all_patients(self, mode: str = \'train\', strict: bool = True):\n        """\n        Load raw data for all patients in the specified version(s).\n        \n        Args:\n            mode: \'train\' or \'test\'\n            strict: Raise when an expected patient file is missing. Set to\n                false only for exploratory partial-dataset inspection.\n            \n        Returns:\n            Dictionary mapping patient IDs to their raw DataFrames\n        """\n        patient_data = {}\n        \n        # Handle both single version and list of versions\n        versions = self.version if isinstance(self.version, list) else [self.version]\n        \n        for version in versions:\n            for patient_id in self.patient_ids[version]:\n                try:\n                    df = self.load_patient_data(patient_id, mode)\n                    patient_data[patient_id] = df\n                    print(f"Successfully loaded patient {patient_id} from version {version}")\n                except FileNotFoundError as e:\n                    if strict:\n                        raise\n                    print(f"Warning: {e}")\n                    continue\n        \n        return patient_data\n    \n    def get_available_patients(self, mode: str = \'train\'):\n        """\n        Get list of available patient IDs for the specified mode.\n        \n        Args:\n            mode: \'train\' or \'test\'\n            \n        Returns:\n            List of available patient IDs\n        """\n        available_patients = []\n        \n        # Handle both single version and list of versions\n        versions = self.version if isinstance(self.version, list) else [self.version]\n        \n        for version in versions:\n            for patient_id in self.patient_ids[version]:\n                file_pattern = f"{patient_id}-ws-{mode}ing.xml"\n                file_path = os.path.join(\n                    self.data_dir, \n                    \'raw\', \n                    \'ohiot1dm\', \n                    version, \n                    mode, \n                    file_pattern\n                )\n                \n                if os.path.exists(file_path):\n                    available_patients.append(patient_id)\n        \n        return available_patients\n\n\ndef load_ohiot1dm_data(data_dir: str, \n                      patient_ids: Optional[List[int]] = None,\n                      mode: str = \'train\',\n                      version: List[str] = [\'2018\', \'2020\'],\n                      sampling_rate: int = 5):\n    """\n    Convenience function to load raw OhioT1DM dataset.\n    \n    Args:\n        data_dir: Path to the data directory\n        patient_ids: List of specific patient IDs to load (None for all)\n        mode: \'train\' or \'test\'\n        version: Dataset version ([\'2018\', \'2020\'])\n        sampling_rate: Target sampling rate in minutes\n        \n    Returns:\n        Dictionary mapping patient IDs to their raw DataFrames\n        \n    Example:\n        >>> data = load_ohiot1dm_data(\'/path/to/data\', patient_ids=[540, 544], mode=\'train\')\n        >>> patient_540_data = data[540]\n    """\n    loader = OhioT1DMDataLoader(data_dir, sampling_rate, version)\n    \n    if patient_ids is None:\n        return loader.load_all_patients(mode)\n    else:\n        patient_data = {}\n        for patient_id in patient_ids:\n            df = loader.load_patient_data(patient_id, mode)\n            patient_data[patient_id] = df\n        return patient_data\n', 'preprocessors.py': '"""\nData preprocessing utilities for blood glucose forecasting.\n\nThis module provides standardized preprocessing functions including:\n- Missing data handling\n- Feature engineering\n- Time series windowing\n"""\n\nimport pandas as pd  # type: ignore\nimport numpy as np\nfrom typing import Dict, List, Tuple, Optional, Union\nimport os\n\n\nclass OhioBGDataPreprocessor:\n    """\n    Comprehensive preprocessor for blood glucose time series data.\n    \n    This class provides standardized preprocessing steps for blood glucose\n    forecasting tasks, ensuring consistent data quality and format across\n    different experiments.\n    \n    This class works specificaly for OhioT1DM database.\n    """\n    \n    def __init__(self, \n                 target_column: str = \'glucose\',\n                 time_column: str = \'index\',\n                 sampling_rate: int = 5):\n        """\n        Initialize the preprocessor.\n        \n        Args:\n            target_column: Name of the glucose column\n            time_column: Name of the time index column\n            sampling_rate: Sampling rate in minutes\n        """\n        self.target_column = target_column\n        self.time_column = time_column\n        self.sampling_rate = sampling_rate\n        \n        # Data quality parameters\n        self.glucose_range = (40, 400)  # Valid glucose range in mg/dL \n        self.max_gap_minutes = 15  # Maximum acceptable gap in minutes\n    \n    def basic_preprocessing(self, df: pd.DataFrame):\n        """\n        Apply basic preprocessing to raw loaded data.\n        \n        Args:\n            df: Raw loaded DataFrame\n            \n        Returns:\n            Basic preprocessed DataFrame\n        """\n        if df.empty:\n            print(f"    [BASIC] Input DataFrame is empty, skipping preprocessing")\n            return df\n\n        print(f"    [BASIC] Starting basic preprocessing on {len(df)} rows")\n        df = df.copy()\n        \n        # Fill missing values with appropriate defaults \n        # Only process columns that exist in the data\n        if \'glucose\' in df.columns:\n            df[\'glucose\'] = df[\'glucose\'].fillna(-1)\n        if \'gsr\' in df.columns:\n            df[\'gsr\'] = df[\'gsr\'].fillna(-1)  # Galvanic Skin Response\n        if \'hr\' in df.columns:\n            df[\'hr\'] = df[\'hr\'].fillna(-1)  # Heart Rate\n        if \'st\' in df.columns:\n            df[\'st\'] = df[\'st\'].fillna(-1)  # Skin Temperature\n        if \'basal\' in df.columns:\n            df[\'basal\'] = df[\'basal\'].ffill()\n        \n        if \'bolus\' in df.columns:\n            df[\'bolus\'] = df[\'bolus\'].fillna(-1)\n        if \'bolus_dur\' in df.columns:\n            df[\'bolus_dur\'] = df[\'bolus_dur\'].fillna(-1)\n        if \'bolus_end\' in df.columns:\n            df[\'bolus_end\'] = df[\'bolus_end\'].fillna(-1)\n        if \'temp_basal\' in df.columns:\n            df[\'temp_basal\'] = df[\'temp_basal\'].fillna(-1)\n        if \'basal_end\' in df.columns:\n            df[\'basal_end\'] = df[\'basal_end\'].fillna(-1)\n        if \'carbs\' in df.columns:\n            df[\'carbs\'] = df[\'carbs\'].fillna(-1)\n        if \'meal_type\' in df.columns:\n            df[\'meal_type\'] = df[\'meal_type\'].fillna(-1)\n        \n        if \'sleep\' in df.columns:\n            df[\'sleep\'] = df[\'sleep\'].fillna(-1)\n        if \'sleep_dur\' in df.columns:\n            df[\'sleep_dur\'] = df[\'sleep_dur\'].fillna(-1)\n        if \'sleep_end\' in df.columns:\n            df[\'sleep_end\'] = df[\'sleep_end\'].fillna(-1)\n        if \'work\' in df.columns:\n            df[\'work\'] = df[\'work\'].fillna(-1)\n        if \'work_dur\' in df.columns:\n            df[\'work_dur\'] = df[\'work_dur\'].fillna(-1)\n        if \'work_end\' in df.columns:\n            df[\'work_end\'] = df[\'work_end\'].fillna(-1)\n        if \'exercise_intensity\' in df.columns:\n            df[\'exercise_intensity\'] = df[\'exercise_intensity\'].fillna(-1)\n        if \'exer_dur\' in df.columns:\n            df[\'exer_dur\'] = df[\'exer_dur\'].fillna(-1)\n        \n        # Drop rows with all NaN values\n        df = df.dropna()\n        \n        # Add helper columns\n        df[\'index\'] = df.index\n        df[\'index_new\'] = df.index\n        \n        # Only create temp columns if the original columns exist\n        if \'bolus\' in df.columns:\n            df[\'temp_bolus\'] = df[\'bolus\']\n        if \'sleep\' in df.columns:\n            df[\'temp_sleep\'] = df[\'sleep\']\n        if \'work\' in df.columns:\n            df[\'temp_work\'] = df[\'work\']\n        if \'exercise_intensity\' in df.columns:\n            df[\'temp_exercise_intensity\'] = df[\'exercise_intensity\']\n        \n        df[\'missing\'] = -1\n        \n        # Remove duplicate rows by taking max values\n        df = df.groupby(df[\'index_new\']).max()\n        \n        # Apply temporal event processing\n        df = self._apply_temporal_events(df)\n        \n        # Check for missing timesteps\n        df = self._check_missing_timesteps(df)\n        \n        # Clean up temporary columns\n        temp_columns = [\'temp_basal\', \'temp_bolus\', \'temp_sleep\', \'temp_work\', \n                       \'temp_exercise_intensity\', \'basal_end\', \'bolus_end\', \n                       \'sleep_end\', \'work_end\', \'exer_dur\', \'missing\']\n        columns_to_drop = [col for col in temp_columns if col in df.columns]\n        if columns_to_drop:\n            df = df.drop(columns=columns_to_drop)\n\n        return df\n    \n    def _apply_temporal_events(self, df: pd.DataFrame):\n        """Apply temporal event durations to the time series."""\n        \n        print(f"        [TEMPORAL] Starting temporal events processing...")\n        \n        # Apply basal insulin rates with proper temporal logic\n        print(f"        [TEMPORAL] Processing basal insulin rates...")\n        df = self._apply_basal_rates(df)\n        \n        # Apply bolus durations\n        if \'temp_bolus\' in df.columns and \'bolus_end\' in df.columns and \'bolus\' in df.columns:\n            for i in range(len(df)):\n                if df[\'temp_bolus\'].iloc[i] != -1:\n                    bolus_end_time = df[\'bolus_end\'].iloc[i]\n                    mask = (df[\'index\'] >= df[\'index\'].iloc[i]) & (df[\'index\'] <= bolus_end_time)\n                    df.loc[mask, \'bolus\'] = df[\'bolus\'].iloc[i]\n            #drop temporary columns\n            df = df.drop(columns=[\'temp_bolus\', \'bolus_end\'], errors=\'ignore\')\n\n        # Apply sleep durations\n        if \'temp_sleep\' in df.columns and \'sleep_end\' in df.columns and \'sleep\' in df.columns:\n            for i in range(len(df)):\n                if df[\'temp_sleep\'].iloc[i] != -1:\n                    sleep_end_time = df[\'sleep_end\'].iloc[i]\n                    mask = (df[\'index\'] >= df[\'index\'].iloc[i]) & (df[\'index\'] <= sleep_end_time)\n                    df.loc[mask, \'sleep\'] = df[\'sleep\'].iloc[i]\n            #drop temporary columns\n            df = df.drop(columns=[\'temp_sleep\', \'sleep_end\'], errors=\'ignore\')\n\n        # Apply work durations\n        if \'temp_work\' in df.columns and \'work_end\' in df.columns and \'work\' in df.columns:\n            for i in range(len(df)):\n                if df[\'temp_work\'].iloc[i] != -1:\n                    work_end_time = df[\'work_end\'].iloc[i]\n                    mask = (df[\'index\'] >= df[\'index\'].iloc[i]) & (df[\'index\'] <= work_end_time)\n                    df.loc[mask, \'work\'] = df[\'work\'].iloc[i]\n\n            #drop temporary columns\n            df = df.drop(columns=[\'temp_work\', \'work_end\'], errors=\'ignore\')\n\n        # Apply exercise durations\n        if (\'temp_exercise_intensity\' in df.columns and \'exer_dur\' in df.columns and \n            \'exercise_intensity\' in df.columns):\n            for i in range(len(df)):\n                if df[\'temp_exercise_intensity\'].iloc[i] != -1:\n                    exercise_duration = df[\'exer_dur\'].iloc[i]\n                    exercise_start = df[\'index\'].iloc[i]\n                    \n                    for j in range(i, len(df)):\n                        time_diff = (df[\'index\'].iloc[j] - exercise_start).total_seconds() / 60.0\n                        if float(exercise_duration) < time_diff:\n                            break\n                        df.iloc[j, df.columns.get_loc(\'exercise_intensity\')] = df[\'exercise_intensity\'].iloc[i]\n            #drop temporary columns\n            df = df.drop(columns=[\'temp_exercise_intensity\', \'exer_dur\'], errors=\'ignore\')\n\n        return df\n    \n    def _apply_basal_rates(self, df: pd.DataFrame):\n        """\n        Apply basal insulin rates with proper temporal logic.\n        \n        This method handles both regular basal rates and temporary basal rates.\n        Basal rates persist until a new rate is set, and temporary basal rates override regular rates for their specified duration.\n\n        Since basal rates are given in hourly units, they are converted to the sampling rate (e.g., 5-minute intervals).\n\n        Args:\n            df: DataFrame with basal rate data\n            \n        Returns:\n            DataFrame with properly applied basal rates\n        """\n        if \'basal\' not in df.columns:\n            print("         [BASAL] No basal column found, skipping basal processing")\n            return df\n            \n        df = df.copy()\n        print(f"            [BASAL] Processing basal rates for {len(df)} rows")\n        \n        # Convert hourly basal rates to sampling rate (e.g., 5-minute rates)\n        # Basal rates in the data are typically in units/hour\n        # Convert to units per sampling interval\n        sampling_factor = self.sampling_rate / 60.0  # Convert minutes to fraction of hour\n        print(f"            [BASAL] Sampling factor: {sampling_factor:.4f} (converting {self.sampling_rate}min intervals)")\n        \n        # Step 1: Handle regular basal rates - forward fill valid values\n        # Replace -1 with NaN for proper forward filling        \n        df[\'basal\'] = df[\'basal\'].replace(-1, np.nan)\n        \n        # Forward fill basal rates (a basal rate persists until changed)\n        df[\'basal\'] = df[\'basal\'].ffill()\n        \n        # Convert hourly rates to sampling interval rates    \n        df[\'basal\'] = pd.to_numeric(df[\'basal\'], errors=\'coerce\')\n        df[\'basal\'] = df[\'basal\'] * sampling_factor\n        # Step 2: Handle temporary basal rates (temp_basal events)\n        if (\'temp_basal\' in df.columns and \'basal_end\' in df.columns):            \n            # Process each temp_basal event\n            for i in range(len(df)):\n                if (df[\'temp_basal\'].iloc[i] != -1 and \n                    not pd.isna(df[\'temp_basal\'].iloc[i]) and\n                    df[\'basal_end\'].iloc[i] != -1 and\n                    not pd.isna(df[\'basal_end\'].iloc[i])):\n                    \n                    temp_basal_rate = pd.to_numeric(df[\'temp_basal\'].iloc[i], errors=\'coerce\')\n                    basal_end_time = df[\'basal_end\'].iloc[i]\n                    temp_start_time = df[\'index\'].iloc[i]\n                    \n                    # Convert temp basal rate to sampling interval rate\n                    temp_basal_rate_interval = temp_basal_rate * sampling_factor\n                    \n                    # Apply temp basal rate from start time to end time\n                    mask = ((df[\'index\'] >= temp_start_time) & \n                           (df[\'index\'] <= basal_end_time))\n                    \n                    df.loc[mask, \'basal\'] = temp_basal_rate_interval\n            \n            # Clean up temporary columns\n            df = df.drop(columns=[\'temp_basal\', \'basal_end\'], errors=\'ignore\')\n        else:\n            print(f"            [BASAL] No temporary basal columns found or incomplete temp basal data")\n        \n        # Fill any remaining NaN values with 0 (no basal insulin)\n        final_nan_count = df[\'basal\'].isna().sum()\n        if final_nan_count > 0:\n            df[\'basal\'] = df[\'basal\'].fillna(0)\n        \n        # Show summary statistics\n        basal_stats = df[\'basal\'].describe()\n        print(f"            [BASAL] Final basal stats - Min: {basal_stats[\'min\']:.4f}, Max: {basal_stats[\'max\']:.4f}, Mean: {basal_stats[\'mean\']:.4f}")\n        \n        return df\n    \n    def _check_missing_timesteps(self, df: pd.DataFrame):\n        """Check for missing timesteps and flag them."""\n        for i in range(1, len(df)):\n            gap = (df[\'index\'].iloc[i] - df[\'index\'].iloc[i-1]).total_seconds() / 60.0 # Convert to minutes\n            if gap != self.sampling_rate:\n                df.iloc[i, df.columns.get_loc(\'missing\')] = gap\n        \n        return df\n\n    def _elapsed_since_last_valid(self, series: pd.Series,\n                                  timestamps: pd.Series):\n        """Minutes from each row back to the most recent observed value."""\n        observed_at = timestamps.where(series.notna()).ffill()\n        return (timestamps - observed_at).dt.total_seconds() / 60.0\n\n    def _gap_span_minutes(self, series: pd.Series,\n                          timestamps: pd.Series):\n        """Total minutes bridged by the gap each row belongs to.\n\n        For a row inside a run of missing values this is the distance from the\n        last observed value to the next one, so a gap is judged as a whole\n        rather than one step at a time. Rows at the edges of the series, where\n        one side has no observation, get infinity and are never filled.\n        """\n        observed_at = timestamps.where(series.notna())\n        before = observed_at.ffill()\n        after = observed_at.bfill()\n        span = (after - before).dt.total_seconds() / 60.0\n        return span.fillna(np.inf)\n\n    def handle_missing_data(self,\n                           df: pd.DataFrame,\n                           strategy: str = \'none\',\n                           max_gap: int = None):\n        """\n        Handle missing data in the time series.\n\n        Gaps are judged in elapsed time, not in row count. The data is in 5 min gaps only where the sensor captured new results, so this function is used to hadle the missing data in between reads.\n\n        Args:\n            df: Input DataFrame\n            strategy: Strategy for handling missing data\n                     (\'none\',\'interpolate\', \'forward_fill\', \'drop\')\n            max_gap: Maximum gap to bridge, in time steps. Defaults to\n                     ``max_gap_minutes // sampling_rate``. Converted to minutes\n                     internally, so it is a real time limit under both a\n                     complete and an interrupted sampling grid.\n\n        Returns:\n            DataFrame with missing data handled\n        """\n        df = df.copy()\n\n        if max_gap is None:\n            max_gap = self.max_gap_minutes // self.sampling_rate\n        if max_gap < 0:\n            raise ValueError(f"max_gap must be non-negative, got {max_gap}")\n        max_gap_minutes = max_gap * self.sampling_rate\n\n        valid_strategies = (\'none\', \'interpolate\', \'forward_fill\', \'drop\')\n        if strategy not in valid_strategies:\n            raise ValueError(\n                f"Unknown missing-data strategy {strategy!r}; expected one of "\n                "\'none\', \'interpolate\', \'forward_fill\', \'drop\'"\n            )\n\n        if self.target_column not in df.columns:\n            print(f"    [MISS] No {self.target_column!r} column present; "\n                  f"strategy {strategy!r} has nothing to act on")\n            return df\n\n        # Convert -1 values to NaN for the target column\n        df[self.target_column] = df[self.target_column].replace(-1, np.nan)\n        before = df[self.target_column].notna().sum()\n        df[self.target_column] = pd.to_numeric(df[self.target_column],\n                                               errors=\'coerce\')\n        unparseable = before - df[self.target_column].notna().sum()\n        if unparseable:\n            print(f"    [MISS] WARNING: {unparseable} non-numeric "\n                  f"{self.target_column} value(s) coerced to NaN")\n        # A -1 written as text survives the replace above, which compares\n        # against the integer sentinel; catch it once the column is numeric.\n        df[self.target_column] = df[self.target_column].replace(-1, np.nan)\n\n        timestamps = pd.to_datetime(\n            df[self.time_column] if self.time_column in df.columns else df.index\n        )\n        timestamps = pd.Series(timestamps.values, index=df.index)\n\n        target = df[self.target_column]\n        gaps_before = int(target.isna().sum())\n        filled = dropped = 0\n\n        if strategy == \'none\':\n            pass\n\n        elif strategy == \'interpolate\':\n            # method=\'time\' weights by real elapsed time, so a step that is\n            # wider than the nominal sampling rate is not treated as one tick.\n            bridgeable = self._gap_span_minutes(target, timestamps) <= max_gap_minutes\n            interpolated = target.interpolate(method=\'time\',\n                                              limit_direction=\'forward\')\n            df[self.target_column] = target.where(~(target.isna() & bridgeable),\n                                                  interpolated)\n            filled = gaps_before - int(df[self.target_column].isna().sum())\n\n        elif strategy == \'forward_fill\':\n            # Carry a value forward only while it is still recent enough; the\n            # elapsed time is measured from the observation itself, so a gap\n            # made of absent rows counts the same as one made of NaN rows.\n            within_limit = self._elapsed_since_last_valid(target, timestamps) <= max_gap_minutes\n            carried = target.ffill()\n            df[self.target_column] = target.where(~(target.isna() & within_limit),\n                                                  carried)\n            filled = gaps_before - int(df[self.target_column].isna().sum())\n\n        elif strategy == \'drop\':\n            # Drop rows with missing glucose values\n            df = df.dropna(subset=[self.target_column])\n            dropped = gaps_before\n\n        remaining = int(df[self.target_column].isna().sum())\n\n        if strategy == \'none\':\n            print(f"    [MISS] Leaving {remaining} gap row(s) as NaN; windows touching them are skipped downstream")\n        elif strategy == \'drop\':\n            print(f"    [MISS] Dropped {dropped} row(s) with a missing "\n                  f"{self.target_column}")\n            print("    [MISS] WARNING: dropping rows closes the gap in the row order without closing it in time, so a downstream window can span an outage without any NaN to stop it; do not report clinical metrics from this run")\n        else:\n            print(f"    [MISS] Using {strategy} strategy with max gap of {max_gap} steps ({max_gap_minutes} min)")\n            print(f"    [MISS] Filled {filled} of {gaps_before} gap row(s); {remaining} left as NaN beyond the {max_gap_minutes}-minute limit")\n            if filled:\n                print("    [MISS] WARNING: filled values become prediction targets and were never measured; do not report clinical metrics from this run")\n        return df\n    \n    def engineer_features(self, df: pd.DataFrame):\n        """\n        Engineer additional features for blood glucose forecasting.\n        \n        Args:\n            df: Input DataFrame\n            \n        Returns:\n            DataFrame with engineered features\n        """\n        df = df.copy()\n        \n        # Ensure glucose column is numeric\n        if self.target_column in df.columns:\n            df[self.target_column] = pd.to_numeric(df[self.target_column], errors=\'coerce\')\n        \n        # Time-based features\n        if self.time_column in df.columns:\n            df[\'hour\'] = df[self.time_column].dt.hour\n            \n            \n            # Cyclical encoding for time features\n            df[\'hour_sin\'] = np.sin(2 * np.pi * df[\'hour\'] / 24)\n            df[\'hour_cos\'] = np.cos(2 * np.pi * df[\'hour\'] / 24)\n\n            print(f"    [ENGINEERING] Engineering features for hour-based cyclicality")\n\n            #drop non-cyclical columns\n            df = df.drop(columns=[\'hour\'], errors=\'ignore\')\n        \n        return df\n    \n    def create_sequences(self, \n                        df: pd.DataFrame,\n                        sequence_length: int = 12,\n                        prediction_horizon: int = 6,\n                        step_size: int = 1):\n        """\n        Create sequences for time series forecasting.\n        \n        Args:\n            df: Input DataFrame\n            sequence_length: Length of input sequences\n            prediction_horizon: Number of steps to predict ahead\n            step_size: Step size for sliding window\n            \n        Returns:\n            Tuple of (X, y) arrays for training\n        """\n        \n        df = df.copy()\n        df[self.target_column] = pd.to_numeric(df[self.target_column], errors=\'coerce\')\n\n        # Drop rows with NaN in target column\n        df = df.dropna(subset=[self.target_column])\n\n        if len(df) < sequence_length + prediction_horizon:\n            raise ValueError("DataFrame too short for specified sequence parameters")\n        \n        # Select numeric columns for features\n        feature_columns = df.select_dtypes(include=[np.number]).columns.tolist()\n        \n        # Remove target column from features to avoid data leakage\n        if self.target_column in feature_columns:\n            feature_columns.remove(self.target_column)\n\n        X, y = [], []\n        \n        for i in range(0, len(df) - sequence_length - prediction_horizon + 1, step_size):\n            # Input sequence\n            x_seq = df.iloc[i:i + sequence_length][feature_columns].values\n            \n            # Target sequence (can be single value or multiple)\n            if prediction_horizon == 1:\n                y_seq = df.iloc[i + sequence_length][self.target_column]\n            else:\n                y_seq = df.iloc[i + sequence_length:i + sequence_length + prediction_horizon][self.target_column].values\n\n            # Check for NaN values\n            if not (np.isnan(x_seq).any() or np.isnan(y_seq).any()):\n                X.append(x_seq)\n                y.append(y_seq)\n        \n        return np.array(X), np.array(y)\n    \n    def preprocess_patient_data(self, \n                               df: pd.DataFrame,\n                               include_feature_engineering: bool = False,\n                               handle_missing: str = \'none\'\n                               ):\n        """\n        Complete preprocessing pipeline for a single patient.\n        \n        Args:\n            df: Input DataFrame for one patient (raw or basic preprocessed)\n            include_feature_engineering: Whether to engineer additional features\n            normalize: Whether to normalize features\n            handle_missing: Strategy for missing data\n            \n        Returns:\n            Fully preprocessed DataFrame\n        """\n        print("Starting preprocessing pipeline...")\n        \n        # Step 0: Apply basic preprocessing if needed (moved from loader)\n        \n        print("0. Applying basic preprocessing...")\n        df = self.basic_preprocessing(df)\n        \n        # Step 1: Handle missing data\n        print("1. Handling missing data...")\n        df = self.handle_missing_data(df, strategy=handle_missing)\n        \n        # Step 2: Feature engineering\n        if include_feature_engineering:\n            print("2. Engineering features...")\n            df = self.engineer_features(df)\n        \n        print("Preprocessing completed!")\n        return df\n\ndef preprocess_ohiot1dm_data(patient_data: Dict[int, pd.DataFrame],\n                            target_column: str = \'glucose\',\n                            **preprocessing_kwargs):\n    """\n    Preprocess data for multiple patients from OhioT1DM dataset.\n    \n    Args:\n        patient_data: Dictionary mapping patient IDs to raw DataFrames\n        target_column: Name of the glucose column\n        **preprocessing_kwargs: Additional arguments for preprocessing\n        \n    Returns:\n        Dictionary of preprocessed DataFrames\n    """\n    preprocessor = OhioBGDataPreprocessor(target_column=target_column)\n    preprocessed_data = {}\n    \n    for patient_id, df in patient_data.items():\n        print(f"\\n=== Preprocessing patient {patient_id} ===")\n        try:\n            processed_df = preprocessor.preprocess_patient_data(\n                df,\n                **preprocessing_kwargs\n            )\n        except Exception as exc:\n            raise RuntimeError(f"Preprocessing failed for patient {patient_id}: {exc}") from exc\n        preprocessed_data[patient_id] = processed_df\n        print(f"Successfully preprocessed patient {patient_id}")\n    \n    return preprocessed_data\n\n\ndef extract_and_save_ohio_data(data_dir: str,\n                         output_dir: str,\n                         patient_ids: Optional[List[int]] = None,\n                         modes: List[str] = [\'train\', \'test\'],\n                         version: str = \'2020\',\n                         sampling_rate: int = 5,\n                         apply_preprocessing: bool = True):\n    """\n    Extract data from XML files and save as CSV files.\n    \n    This function loads raw data and optionally applies preprocessing\n    before saving to CSV files.\n    \n    Args:\n        data_dir: Path to the data directory containing XML files\n        output_dir: Directory to save extracted CSV files\n        patient_ids: List of specific patient IDs to process (None for all)\n        modes: List of modes to process (\'train\', \'test\')\n        version: Dataset version (\'2018\' or \'2020\')\n        sampling_rate: Target sampling rate in minutes\n        apply_preprocessing: Whether to apply preprocessing before saving\n    """\n    from .loaders import OhioT1DMDataLoader  # Import here to avoid circular imports\n    \n    loader = OhioT1DMDataLoader(data_dir, sampling_rate, version)\n    \n    # Create output directory if it doesn\'t exist\n    os.makedirs(output_dir, exist_ok=True)\n    \n    if patient_ids is None:\n        patient_ids = loader.patient_ids[version]\n    \n    if apply_preprocessing:\n        preprocessor = OhioBGDataPreprocessor()\n    \n    for mode in modes:\n        print(f"\\n=== Processing {mode} data for version {version} ===")\n        \n        for patient_id in patient_ids:\n            try:\n                print(f"\\nProcessing patient {patient_id}...")\n                df = loader.load_patient_data(patient_id, mode)\n                \n                if not df.empty:\n                    if apply_preprocessing:\n                        # Apply basic preprocessing\n                        df = preprocessor.basic_preprocessing(df)\n                    \n                    suffix = "_processed" if apply_preprocessing else "_raw"\n                    output_file = os.path.join(output_dir, f"{patient_id}_{mode}{suffix}.csv")\n                    df.to_csv(output_file)\n                    print(f"Saved data for patient {patient_id} to {output_file}")\n                else:\n                    print(f"No data available for patient {patient_id}")\n                    \n            except FileNotFoundError as e:\n                print(f"Warning: Could not process patient {patient_id}: {e}")\n                continue\n'}
SUPPORT_DIR = LOCAL_ROOT / 'code'
SUPPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, source in BUNDLE.items():
    (SUPPORT_DIR / name).write_text(source)
spec = importlib.util.spec_from_file_location('cold_start', SUPPORT_DIR / 'run_cold_start.py')
cold = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cold)
import copy
CFG = copy.deepcopy(cold.DEFAULTS)
assert HORIZON_MINUTES > 0 and HORIZON_MINUTES % 5 == 0
CFG.update(model=MODEL, horizon_steps=HORIZON_MINUTES//5,
           budgets_days=BUDGETS_DAYS, seeds=SEEDS, smoke=SMOKE_TEST)
if SMOKE_TEST:
    CFG.update(patients=[559, 563], seeds=[41], budgets_days=[3, 'full'],
               hidden_size=8, layers=1, source_epochs=1, target_epochs=1,
               bootstrap_replicates=199)
print(json.dumps(CFG, indent=2))
print(f"Plan: {len(CFG['patients'])*len(CFG['seeds'])} source pretraining fits, "
      f"{2*len(CFG['patients'])*len(CFG['seeds'])*len(CFG['budgets_days'])} target fits.")


## 4. Stage data from Drive

Only the patients in this configuration are staged. Copies are checked against Drive by SHA-256 so stale local files are not silently reused. Data remain in the private runtime/Drive paths. Ensure your existing use of Drive complies with your data agreement.


In [ ]:
DATA_ROOT = LOCAL_ROOT / 'data'
for release, patients in cold.COHORT.items():
    for pid in patients:
        if pid not in CFG['patients']:
            continue
        for mode in ['train', 'test']:
            name = f'{pid}-ws-{mode}ing.xml'
            src = DRIVE_DATA / release / mode / name
            dst = DATA_ROOT / 'raw' / 'ohiot1dm' / release / mode / name
            if not src.is_file():
                raise FileNotFoundError(f'Missing {src}; use the same Drive dataset as the main notebook.')
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or cold.digest(src) != cold.digest(dst):
                shutil.copy2(src, dst)
            assert cold.digest(src) == cold.digest(dst)
print('All required XML files staged and verified.')


## 5. Preflight, train, evaluate and resume

Before any GPU training, **every patient/budget is checked** for enough contiguous fitting/validation/test windows and a fixed test set. If a budget is infeasible, the run stops with `preflight.json`; do not silently drop that patient. Inspect coverage before deciding on a revised protocol, and document the change.

Training prints progress every epoch. A completed source checkpoint is cached once per target/seed. Each paired RL/TL budget job is copied to Drive, with its completion marker written last. A disconnect can require repeating the currently unfinished source or paired target job, but completed work is reused.

After reconnecting, run the notebook again with identical settings. Different data, code, configurations or recorded package versions use a separate fingerprinted folder. Avoid simultaneous sessions writing the same run.


In [ ]:
import time
started = time.perf_counter()
LOCAL_RUN, DRIVE_RUN = cold.run(
    DATA_ROOT, SUPPORT_DIR, CFG,
    local_root=LOCAL_ROOT / 'runs',
    drive_root=DRIVE_RESULTS / 'cold_start_followup',
    device=DEVICE)
print(f'Finished in {(time.perf_counter()-started)/3600:.2f} hours.')
print('Results:', DRIVE_RUN)


## 6. Review all budgets

- `summary/budget_summary.csv`: equal-patient MAE/RMSE, transfer benefit, joint patient-and-seed intervals, and the additional benefit relative to this experiment's full-history control.
- `summary/patient_seed_metrics.csv`: each patient's results, sequence counts and persistence reference.
- `summary/history_budget_curve.png`: all budgets with pointwise confidence intervals.
- `preflight.json`: actual history intervals, usable windows, normalization statistics and test dates.
- `jobs/`: predictions and validation histories for both regimes.
- `pretraining/`: source-only checkpoints, donor IDs and histories.
- `protocol.json`: exact configuration and data/code fingerprints.

Confidence intervals resample patients and shared seeds, using identical draws for all budgets. Wilcoxon tests use 12 patient means, with BH across both metrics and all budgets in this run; budgets are not independent replications. If adding architectures/horizons, combine the testing family before making joint significance claims.

**Interpretation:** stronger benefit with short history would support a cold-start contribution. It does not establish that every new patient benefits, that full-history gains were underestimated, or that MAE and RMSE percentages are interchangeable. Inspect absolute errors, persistence and seed variability. Report null/negative results as well as improvements.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
display(Markdown((LOCAL_RUN / 'summary' / 'REPORT.md').read_text()))
display(pd.read_csv(LOCAL_RUN / 'summary' / 'budget_summary.csv'))
display(Image(filename=str(LOCAL_RUN / 'summary' / 'history_budget_curve.png')))
metrics = pd.read_csv(LOCAL_RUN / 'summary' / 'patient_seed_metrics.csv')
display(metrics.groupby('budget_days')[['n_train', 'n_validation', 'n_test']].agg(['min','median','max']))
print('Download article candidates from:', DRIVE_RUN / 'summary')
